In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
# import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio
# from astropy.io import fits

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
cat = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/fugu/main_cumulative_lrg.fits'))
cat['EFFTIME_ELG'] = 8.60 * cat['TSNR2_ELG']
cat['EFFTIME_LRG'] = 12.15 * cat['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = cat['COADD_FIBERSTATUS']==0
print('FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Remove "no data" fibers
mask = cat['ZWARN'] & 2**9==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Apply LRG mask
mask = cat['lrg_mask']==0
print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# # Remove QSO targets
# mask = cat['DESI_TARGET'] & 2**2 ==0
# print('Remove QSO targets', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
# cat = cat[mask]

# Require a minimum depth
min_depth = 800.
mask = cat['EFFTIME_LRG']>min_depth
print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
cat = cat[mask]

# # Julien's bad fibers list + my list of worst-performing fibers; bad_fibers-everest.ipynb
# # bad_fibers = np.loadtxt('/global/cfs/cdirs/desi/users/rongpu/spectro/everest/misc/bad_fibers_20211117.txt', dtype=int)
# bad_fibers = np.loadtxt('/Users/rongpu/Documents/Data/desi_data/everest/misc/bad_fibers_20211117.txt', dtype=int)
# print(len(bad_fibers))
# mask_bad = np.in1d(cat['FIBER'], bad_fibers)
# print('Bad fibers', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
# cat = cat[~mask_bad]
# print(len(cat), len(np.unique(cat['TARGETID'])))

FIBERSTATUS 338266 7165 0.020742203218587794
No data 338265 1 2.9562533627382e-06
LRG mask 304337 33928 0.1003000606033731
Min depth 292792 11545 0.9620650791721019


In [4]:
# Custom DELTACHI2 vs z cut
d = (10**(3 - 3.5*cat['Z']))
mask_remove = (d>30) & (cat['DELTACHI2']<30)
mask_remove |= (d<30) & (cat['DELTACHI2']<d)
mask_remove |= (cat['DELTACHI2']<10)
mask_quality = cat['ZWARN']==0
mask_quality &= cat['Z']<1.4
mask_quality &= (~mask_remove)

print(np.sum(~mask_quality)/len(mask_quality))
cat = cat[mask_quality]
print(len(cat))

0.013238066613841908
288916


In [5]:
mask_star = (cat['SPECTYPE']=='STAR') | (cat['Z']<0.0003)
cat['SPECTYPE'][mask_star] = 'STAR'
print(np.sum(mask_star)/len(mask_star))

0.004738401473092525


In [6]:
t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'], return_counts=True)
t['frac (%)'] = t['count']/len(cat)*100
t['frac (%)'].format = '%.2f'
t.sort('count')
t

SPECTYPE,count,frac (%)
str6,int64,float64
STAR,1369,0.47
QSO,5619,1.94
GALAXY,281928,97.58


In [7]:
# Fraction of QSO targets
mask = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask)/len(mask))

0.013429508923008765


In [8]:
# Exclude QSO targets
mask = cat['DESI_TARGET'] & 2**2==0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.2f'
t.sort('count')
t

285036 0.9865704910769912


SPECTYPE,count,frac (%)
str6,int64,float64
STAR,1291,0.45
QSO,3063,1.07
GALAXY,280682,98.47


In [9]:
# Select QSO targets
mask = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.2f'
t.sort('count')
t

3880 0.013429508923008765


SPECTYPE,count,frac (%)
str6,int64,float64
STAR,78,2.01
GALAXY,1246,32.11
QSO,2556,65.88


In [10]:
# Downweight QSO targets and recalculate the fractions
mask_qso = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask_qso)/len(mask_qso))
cat['weight'] = 1.
cat['weight'][mask_qso] = 0.5

mask = cat['SPECTYPE']=='STAR'
print('Star fraction: {:.2f}%'.format(100*np.sum(mask*cat['weight'])/np.sum(cat['weight'])))
mask = cat['SPECTYPE']=='QSO'
print('QSO fraction: {:.2f}%'.format(100*np.sum(mask*cat['weight'])/np.sum(cat['weight'])))

0.013429508923008765
Star fraction: 0.46%
QSO fraction: 1.51%


------
# Spectype of masked LRG targets

In [11]:
cat = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/fugu/main_cumulative_lrg.fits'))
cat['EFFTIME_ELG'] = 8.60 * cat['TSNR2_ELG']
cat['EFFTIME_LRG'] = 12.15 * cat['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = cat['COADD_FIBERSTATUS']==0
print('FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Remove "no data" fibers
mask = cat['ZWARN'] & 2**9==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Apply LRG mask
mask = cat['lrg_mask']!=0
print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# # Remove QSO targets
# mask = cat['DESI_TARGET'] & 2**2 ==0
# print('Remove QSO targets', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
# cat = cat[mask]

# Require a minimum depth
min_depth = 800.
mask = cat['EFFTIME_LRG']>min_depth
print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
cat = cat[mask]

# # Julien's bad fibers list + my list of worst-performing fibers; bad_fibers-everest.ipynb
# # bad_fibers = np.loadtxt('/global/cfs/cdirs/desi/users/rongpu/spectro/everest/misc/bad_fibers_20211117.txt', dtype=int)
# bad_fibers = np.loadtxt('/Users/rongpu/Documents/Data/desi_data/everest/misc/bad_fibers_20211117.txt', dtype=int)
# print(len(bad_fibers))
# mask_bad = np.in1d(cat['FIBER'], bad_fibers)
# print('Bad fibers', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
# cat = cat[~mask_bad]
# print(len(cat), len(np.unique(cat['TARGETID'])))

FIBERSTATUS 338266 7165 0.020742203218587794
No data 338265 1 2.9562533627382e-06
LRG mask 33928 304337 0.899699939396627
Min depth 32646 1282 0.9622141004480076


In [12]:
# Custom DELTACHI2 vs z cut
d = (10**(3 - 3.5*cat['Z']))
mask_remove = (d>30) & (cat['DELTACHI2']<30)
mask_remove |= (d<30) & (cat['DELTACHI2']<d)
mask_remove |= (cat['DELTACHI2']<10)
mask_quality = cat['ZWARN']==0
mask_quality &= cat['Z']<1.4
mask_quality &= (~mask_remove)

print(np.sum(~mask_quality)/len(mask_quality))
cat = cat[mask_quality]
print(len(cat))

0.023034981314709307
31894


In [13]:
mask_star = (cat['SPECTYPE']=='STAR') | (cat['Z']<0.0003)
cat['SPECTYPE'][mask_star] = 'STAR'
print(np.sum(mask_star)/len(mask_star))

0.10688530758136326


In [14]:
t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'], return_counts=True)
t['frac (%)'] = t['count']/len(cat)*100
t['frac (%)'].format = '%.1f'
t.sort('count')
t

SPECTYPE,count,frac (%)
str6,int64,float64
QSO,622,2.0
STAR,3409,10.7
GALAXY,27863,87.4


In [15]:
# Fraction of QSO targets
mask = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask)/len(mask))

0.019846993164858596


In [16]:
# Exclude QSO targets
mask = cat['DESI_TARGET'] & 2**2==0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.1f'
t.sort('count')
t

31261 0.9801530068351414


SPECTYPE,count,frac (%)
str6,int64,float64
QSO,327,1.0
STAR,3235,10.3
GALAXY,27699,88.6


In [17]:
# Select QSO targets
mask = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.1f'
t.sort('count')
t

633 0.019846993164858596


SPECTYPE,count,frac (%)
str6,int64,float64
GALAXY,164,25.9
STAR,174,27.5
QSO,295,46.6


In [18]:
# Downweight QSO targets and recalculate the fractions
mask_qso = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask_qso)/len(mask_qso))
cat['weight'] = 1.
cat['weight'][mask_qso] = 0.3

mask = cat['SPECTYPE']=='STAR'
print('Star fraction: {:.2f}%'.format(100*np.sum(mask*cat['weight'])/np.sum(cat['weight'])))
mask = cat['SPECTYPE']=='QSO'
print('QSO fraction: {:.2f}%'.format(100*np.sum(mask*cat['weight'])/np.sum(cat['weight'])))

0.019846993164858596
Star fraction: 10.45%
QSO fraction: 1.32%
